# 第13回：回帰分析（最小二乗法・相関係数）

今回は、気象庁が公開している日本沿岸域の海面水温データを使って、
水温の長期的な変化傾向を**客観的・量的に評価する**方法を学びます。

前半の実験でも最小二乗法を扱っています。
ここではPythonを使いながら、最小二乗法が何をしているのかを実際に確認します。

### このNotebookの進め方

このNotebookでは、**上から順番にCodeセルを実行してください**。
前のセルで作成した変数や読み込んだデータを、後のセルでも使います。

Codeセルを選択して **Shift + Enter** を押すと、そのセルを実行して次のセルへ進みます。

### 目標
- 水温の変化傾向を量的に計算できる
- 時間と水温の関係を相関係数によって量的に示すことができる

## 1. 水温データをダウンロードする

気象庁「日本沿岸域の海面水温情報」から、授業で使用する海域のデータを取得します。

https://www.data.jma.go.jp/kaiyou/data/db/kaikyo/series/engan/engan.html

1. 上のページを開く
2. **近畿・中国・四国**を選択する
3. **豊後水道南部**を選択する
4. ページ左下の「データ」のリンクからデータを保存する

この授業では、保存したファイルを **`area518.txt`** とします。

### JupyterLiteにデータを置く

ダウンロードした `area518.txt` を、JupyterLite左側のファイル一覧へドラッグ＆ドロップしてください。

このNotebookと**同じフォルダ**に `area518.txt` がある状態にしておくと、

```python
pd.read_csv("area518.txt")
```

のように、ファイル名だけで読み込むことができます。

## 2. pandasでデータを読み込む

まず、必要なライブラリを読み込みます。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

`area518.txt` を読み込みます。

ファイルの拡張子は `.txt` ですが、中身は値がカンマで区切られた表形式のデータです。
そのため、`pd.read_csv()` で読み込むことができます。

In [ ]:
df = pd.read_csv("area518.txt")
df

### 元のデータとDataFrameを比べる

元のテキストファイルでは、1行に複数の値がカンマで区切られて並んでいます。

pandasで読み込むと、それぞれが `yyyy`, `mm`, `dd`, `areaNo.`, `flag`, `Temp.` などの**列**として整理された `DataFrame` になります。

まず、最初の数行を確認しましょう。

In [ ]:
df.head()

列名とデータの大きさも確認します。

In [ ]:
print(df.columns)
print(df.shape)

### 列を取り出す

準備2でも扱ったように、DataFrameでは列名を指定して、その列を取り出すことができます。

例えば、年を表す `yyyy` と水温を表す `Temp.` を取り出してみます。

In [ ]:
df["yyyy"]

In [ ]:
df["Temp."]

複数の列を同時に取り出すこともできます。

In [ ]:
df[["yyyy", "mm", "dd", "Temp."]].head()

## 3. 日付を表す変数を作る

元のデータには `date` という列はなく、年・月・日が `yyyy`, `mm`, `dd` に分かれて入っています。

そこで、この3つの列から日付を表す新しい列 `date` を作ります。

In [ ]:
df["date"] = pd.to_datetime(
    dict(year=df["yyyy"], month=df["mm"], day=df["dd"])
)

df.head()

右端に `date` が追加されたことを確認してください。

## 4. 水温の時間変化を図にする

作成した `date` を横軸、水温 `Temp.` を縦軸にして、日々の水温変化を表示します。

In [ ]:
plt.plot(df["date"], df["Temp."])
plt.xlabel("Date")
plt.ylabel("Temperature")
plt.show()

日々の変動が大きく、このままでは長期的な変化傾向が分かりにくいことが分かります。

そこで、年ごとの平均水温を計算します。

## 5. 年平均水温を計算する

準備2では、列全体の平均値を

```python
df["Temp."].mean()
```

のように求めました。

今回は、同じ年のデータをひとつのグループとしてまとめ、それぞれについて平均を求めます。
そのために `groupby()` を使います。

In [ ]:
annual = df.groupby("yyyy")["Temp."].mean()
annual.head()

`groupby("yyyy")` で同じ年のデータをまとめ、その各グループについて `Temp.` の `mean()` を計算しています。

年平均水温を図にしてみましょう。

In [ ]:
plt.plot(annual.index, annual.values, marker="o")
plt.xlabel("Year")
plt.ylabel("Annual mean temperature")
plt.show()

日々のデータよりも、長期的な変化が見やすくなりました。

では、この水温がどの程度変化しているのかを数値で表してみましょう。

## 6. 最小二乗法

年平均水温を直線

$$y=ax+b$$

で近似することを考えます。$a$ は傾き、$b$ は切片です。

「データに最もよく合う直線」は、どのように決めればよいでしょうか。

### 平均からの偏差を使う

まず、年と水温からそれぞれの平均値を引きます。

In [ ]:
x = annual.index.to_numpy()
y = annual.to_numpy()
xa = x - x.mean()
ya = y - y.mean()

print("xa の平均 =", xa.mean())
print("ya の平均 =", ya.mean())

平均を引いたので、

$$\bar{x}=0,\qquad \bar{y}=0$$

となっています。

ただし、**平均を引いたからといって、最初から $b=0$ と決めてよいわけではありません。**
まずは $y=ax+b$ のまま、どの $a,b$ がデータによく合うかを調べます。

### 適当な直線を引いて、誤差を測る

まず、`a` と `b` を適当に決めて直線を引いてみます。

In [ ]:
a = 0.02
b = 0.01
y_fit = a * xa + b
error = ((ya - y_fit)**2).sum()

plt.plot(xa, ya, "o", label="Data")
plt.plot(xa, y_fit, label="y = ax + b")
for xi, yi, yfi in zip(xa, ya, y_fit):
    plt.plot([xi, xi], [yfi, yi], linewidth=0.7)

j = len(xa)//2
xi, yi, yfi = xa[j], ya[j], y_fit[j]
plt.annotate("", xy=(xi, yi), xytext=(xi, yfi),
             arrowprops=dict(arrowstyle="<->"))
plt.text(xi+1, (yi+yfi)/2, r"$e_i=y_i-\hat{y}_i$")
plt.xlabel("Year anomaly")
plt.ylabel("Temperature anomaly")
plt.legend()
plt.show()

print("a =", a, " b =", b)
print("残差平方和 =", error)

図の縦線は、観測値と直線の**xを固定したときのy方向の残差**を表します。

観測値を $(x_i,y_i)$、同じ $x_i$ における直線上の値を $(x_i,\hat y_i)$ とすると、

$$e_i=y_i-\hat y_i$$

です。最小二乗法では、

$$E=\sum_i e_i^2=\sum_i\{y_i-(ax_i+b)\}^2$$

が最も小さくなる $a,b$ を求めます。

`a` や `b` を変えて、直線と残差平方和がどう変わるか試してみましょう。

In [ ]:
a = 0.03  # 変えてみる
b = 0.05  # 変えてみる
y_fit = a * xa + b
error = ((ya - y_fit)**2).sum()

plt.plot(xa, ya, "o")
plt.plot(xa, y_fit)
for xi, yi, yfi in zip(xa, ya, y_fit):
    plt.plot([xi, xi], [yfi, yi], linewidth=0.7)
plt.xlabel("Year anomaly")
plt.ylabel("Temperature anomaly")
plt.show()

print("a =", a, " b =", b)
print("残差平方和 =", error)

### まず切片 b を動かしてみる

傾き `a` を固定して、切片 `b` を少しずつ変え、それぞれの残差平方和を図にします。

In [ ]:
a = 0.02
blist = np.arange(-0.5, 0.51, 0.01)
errors_b = []
for b in blist:
    errors_b.append(((ya - (a*xa+b))**2).sum())

plt.plot(blist, errors_b)
plt.xlabel("Intercept b")
plt.ylabel("Sum of squared residuals")
plt.show()

残差平方和は $b$ に対して**放物線（二次関数）**になり、平均を引いたデータでは一番低いところが $b=0$ になります。

したがって最適な切片は

$$b=0$$

です。なぜそうなるのかを補足で確認します。

### 補足：平均を引くと、なぜ最適な b は 0 になるのか

傾き $a$ を固定すると、

$$
E(b)=\sum_i(y_i-ax_i-b)^2
$$

です。展開すると、

$$
E(b)
=
Nb^2
-2b\sum_i(y_i-ax_i)
+\sum_i(y_i-ax_i)^2
$$

となります。

平均を引いたデータでは、

$$
\sum_i x_i=0,\qquad \sum_i y_i=0
$$

なので、

$$
\sum_i(y_i-ax_i)
=
\sum_i y_i-a\sum_i x_i
=0
$$

です。

したがって、

$$
E(b)
=
Nb^2+\sum_i(y_i-ax_i)^2
=
Nb^2+\text{（$b$ によらない部分）}
$$

となります。

第2項は $b$ によって変化しません。また、第1項 $Nb^2$ は $b=0$ のときに最小になります。

したがって、残差平方和が最も小さくなる切片は

$$
b=0
$$

です。

このため、平均を引いた後の最小二乗直線は

$$
y=ax
$$

として考えることができます。

### 次に傾き a を動かしてみる

`b = 0` として、傾き `a` を少しずつ変えます。

In [ ]:
b = 0.0
alist = np.arange(-0.05, 0.10, 0.001)
errors_a = []
for a in alist:
    errors_a.append(((ya - (a*xa+b))**2).sum())

plt.plot(alist, errors_a)
plt.xlabel("Slope a")
plt.ylabel("Sum of squared residuals")
plt.show()

ここでも残差平方和は $a$ に対して放物線になります。したがって、**放物線の一番低いところ**に対応する $a$ が最適な傾きです。

残差平方和

$$
E(a)=\sum_i(y_i-ax_i)^2
$$

を展開すると、

$$
E(a)
=
\left(\sum_i x_i^2\right)a^2
-2\left(\sum_i x_i y_i\right)a
+\sum_i y_i^2
$$

となり、$a$ の二次関数になっています。

これを平方完成すると、

$$
E(a)
=
\left(\sum_i x_i^2\right)
\left(
a-\frac{\sum_i x_i y_i}{\sum_i x_i^2}
\right)^2
+
\sum_i y_i^2
-
\frac{\left(\sum_i x_i y_i\right)^2}
{\sum_i x_i^2}
$$

となります。

後ろの2項は $a$ によって変化しません。また、第1項は0以上なので、

$$
a=
\frac{\sum_i x_i y_i}
{\sum_i x_i^2}
$$

のときに $E(a)$ は最小になります。

したがって、この $a$ が最小二乗法で求める直線の傾きです。

In [ ]:
a = (xa * ya).sum() / (xa * xa).sum()
b = 0.0
print("a =", a)
print("b =", b)

In [ ]:
plt.plot(xa, ya, "o", label="Data")
plt.plot(xa, a*xa+b, label="Least-squares line")
plt.xlabel("Year anomaly")
plt.ylabel("Temperature anomaly")
plt.legend()
plt.show()

print("1年あたりの変化 =", a)
print("10年間あたりの変化 =", a*10)

### 補足：微分を使って求めると？

平均を引く前の一般的な直線 $y=ax+b$ について、

$$E(a,b)=\sum_i\{y_i-(ax_i+b)\}^2$$

を最小にします。

まず $b$ について、

$$
\frac{\partial E}{\partial b}
=-2\sum_i(y_i-ax_i-b)=0
$$

より、

$$b=\bar y-a\bar x$$

となります。**一般には $b=0$ ではありません。**

今回のように平均を引いた後では $\bar x=0,\ \bar y=0$ なので、

$$b=0$$

となります。

次に $a$ について、

$$
\frac{\partial E}{\partial a}
=-2\sum_i x_i(y_i-ax_i-b)=0.
$$

平均を引いたデータでは $b=0$ なので、

$$
a=\frac{\sum_i x_i y_i}{\sum_i x_i^2}
$$

となり、放物線の軸から求めた結果と一致します。

### NumPyで計算する

最後に `np.polyfit()` で計算し、自分で求めた傾きと一致することを確認します。

In [ ]:
a_np, b_np = np.polyfit(x, y, 1)
print("自分で計算した傾き =", a)
print("np.polyfitの傾き    =", a_np)
print("np.polyfitの切片    =", b_np)

### ここまでで課題1に取り組めます

ここまでで、年平均水温を求め、最小二乗法を使って水温の変化傾向を計算できるようになりました。

`exercise13.ipynb` の **課題1** に進んでください。

課題1が終わったら、このNotebookに戻り、次の「相関係数」へ進みます。

## 7. 相関係数

次に、**時間と水温がどの程度関係しているか**を数値で表します。

平均からの偏差 `xa`, `ya` を使うと、相関係数は

$$
r=
\frac{
\frac{1}{N}\sum_i x_i y_i
}{
\sqrt{\frac{1}{N}\sum_i x_i^2}
\sqrt{\frac{1}{N}\sum_i y_i^2}
}
$$

と表せます。

分母は、それぞれの変数の**データの広がり（標準偏差）**を表しています。
それぞれを自分自身の標準偏差で割ることで、**x方向とy方向のデータの広がりを同じくらいにそろえる**ことができます。
これによって、年と水温のように単位や値の大きさが異なる変数でも、関係の強さを同じ尺度で表すことができます。

- $r$ が 1 に近い：強い正の相関
- $r$ が -1 に近い：強い負の相関
- $r$ が 0 に近い：線形な関係が弱い

まず、式をそのままPythonで計算します。

In [ ]:
N = len(xa)

r = ((xa * ya).sum() / N) / (
    np.sqrt((xa**2).sum() / N) *
    np.sqrt((ya**2).sum() / N)
)

print("相関係数 =", r)

pandasやNumPyの機能でも計算できます。自分で計算した値と一致することを確認しましょう。

In [ ]:
print("自分で計算 :", r)
print("pandas     :", pd.Series(xa).corr(pd.Series(ya)))
print("NumPy      :", np.corrcoef(xa, ya)[0, 1])

### ここまでで課題2に取り組めます

ここまでで、2つの変数の関係を相関係数で表せるようになりました。

`exercise13.ipynb` の **課題2** に進んでください。

課題2が終わったら、続けて課題3に進みます。

### 課題3へ

課題3では、好きな2つの地域を選び、これまでに扱った**相関係数と最小二乗法**を組み合わせて、2地域の年平均水温の関係を調べます。

`exercise13.ipynb` の **課題3** に進んでください。

## 補足：データの広がりをそろえるとは？

相関係数の分母にある

$$
\sigma_x=\sqrt{\frac{1}{N}\sum_i x_i^2},
\qquad
\sigma_y=\sqrt{\frac{1}{N}\sum_i y_i^2}
$$

は、それぞれのデータの標準偏差です。

そこで、

$$
X_i=\frac{x_i}{\sigma_x},
\qquad
Y_i=\frac{y_i}{\sigma_y}
$$

とすると、それぞれのデータの標準偏差は1になります。

つまり、標準偏差で割ることは、**x方向とy方向のデータの広がりを、それぞれ「標準偏差 = 1」になるようにそろえる**操作と考えることができます。

このように、データの大きさや広がりを一定の基準にそろえることを**規格化（normalization）**と呼ぶことがあります。

In [ ]:
sx = np.sqrt(np.mean(xa**2))
sy = np.sqrt(np.mean(ya**2))

X = xa / sx
Y = ya / sy

print("標準偏差で割る前")
print("x:", np.std(xa), " y:", np.std(ya))

print("標準偏差で割った後")
print("X:", np.std(X), " Y:", np.std(Y))

標準偏差で割る前と後の散布図を比べてみます。

In [ ]:
plt.scatter(xa, ya)
plt.axhline(0, linewidth=0.7)
plt.axvline(0, linewidth=0.7)
plt.xlabel("Year anomaly")
plt.ylabel("Temperature anomaly")
plt.title("Before dividing by standard deviation")
plt.show()

plt.scatter(X, Y)
plt.axhline(0, linewidth=0.7)
plt.axvline(0, linewidth=0.7)
plt.xlabel("Normalized year anomaly")
plt.ylabel("Normalized temperature anomaly")
plt.title("After dividing by standard deviation")
plt.show()

標準偏差で割った後は、x方向・y方向ともデータの標準偏差が1になっています。

このとき相関係数は、

$$
r=\frac{1}{N}\sum_i X_iY_i
$$

と書けます。

したがって相関係数は、**それぞれの変数の広がりをそろえたうえで、2つの変数が同じ向きに変化する傾向を表した量**と考えることができます。

## 8. まとめ

今回は、

- pandasで実際の水温データを読み込み、表形式のデータとして扱う
- 年平均水温を計算する
- 最小二乗法が「y方向の残差平方和」を最小にすることを確認する
- 最小二乗法を式から計算し、NumPyの結果と比較する
- 相関係数を式から計算し、pandas・NumPyの結果と比較する

という流れで、水温の長期変化を量的に評価しました。

次は、別の海域のデータについて自分で同じ解析を行います。